In [ ]:
#pip install pandas
#pip install psycopg2

In [ ]:
import pandas as pd
import psycopg2
import hashlib

In [ ]:
df = pd.read_csv('nba_teams.csv', delimiter=',' )
conn = psycopg2.connect(
    dbname="nba_sql4",
    user="postgres",
    password="1234",
    host="localhost",
    port="5432"
)
cur = conn.cursor()
SEASON_ID = 1

In [ ]:
index = 1
for _, row in df.iterrows():

    try:
        # Insert into Player
        cur.execute("""
            INSERT INTO Team (team_id, name)
            VALUES (%s, %s) 
            """, (index,row['Team']))

        #print(f\"Inserted {team_name}\")
        index = index+1
        
    except Exception as e:
        print(f"Error inserting {row['Team']}: {e}")

conn.commit()

In [ ]:
df = pd.read_csv('Player_Season_stats.csv', delimiter=';' , encoding="cp1252")
    
# --- Keep only first entry for each player (skip TOT/MIN duplicates) ---
df = df.drop_duplicates(subset=['Player'], keep='first')

In [ ]:
# --- Database connection ---
conn = psycopg2.connect(
    dbname="nba_sql4",
    user="postgres",
    password="1234",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# --- Fixed season ID ---
SEASON_ID = 1

In [ ]:
# --- Insert players and their season stats ---
for idx, row in df.iterrows():
    name = row['Player']
    team_id = row['Tm']
    
    # Optional: use hashed player_id
    player_id = int(hashlib.md5(name.encode()).hexdigest(), 16) % (10 ** 8)

    pts = float(row['PTS'])
    trb = float(row['TRB'])
    ast = float(row['AST'])
    fg_pct = float(row['FG%']) if not pd.isna(row['FG%']) else None
    fg3_pct = float(row['3P%']) if not pd.isna(row['3P%']) else None
    ft_pct = float(row['FT%']) if not pd.isna(row['FT%']) else None
    efg_pct = float(row['eFG%']) if not pd.isna(row['eFG%']) else None

    try:
        # Insert into Player
        cur.execute("""
            INSERT INTO Player (player_id, name, team_id)
            VALUES (%s, %s, %s)
            ON CONFLICT (player_id) DO NOTHING;
        """, (player_id, name, team_id))

        # Insert into Player_Season
        cur.execute("""
            INSERT INTO Player_Season (player_id, season_id, pts, trb, ast, fg_pct, fg3_pct, ft_pct, efg_pct)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (player_id, season_id) DO NOTHING;
        """, (player_id, SEASON_ID, pts, trb, ast, fg_pct, fg3_pct, ft_pct, efg_pct))
        
        print(f"Inserted {name}")
        
    except Exception as e:
        print(f"Error inserting {name}: {e}")

# --- Commit and close ---
conn.commit()
cur.close()
conn.close()

In [ ]:
df = pd.read_csv('games_data_2022_23.csv', delimiter=',')

df['Home_Neutral']

In [ ]:
# --- Database connection ---
conn = psycopg2.connect(
    dbname="nba_sql4",
    user="postgres",
    password="1234",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# --- Fixed season ID ---
SEASON_ID = 1

In [ ]:
game_id = 22200001
for idx,row in df.iterrows():
    home_team_name = row['Home_Neutral']
    away_team_name = row['Visitor_Neutral']

    score_home_team = int(row['PTS_Home'])
    score_away_team = int(row['PTS_Visitor'])

    try:
    # Get team IDs properly (fetch result after execution)
        cur.execute("SELECT team_id FROM team WHERE name = %s;", (home_team_name,))
        home_team_id = cur.fetchone()
    
        cur.execute("SELECT team_id FROM team WHERE name = %s;", (away_team_name,))
        away_team_id = cur.fetchone()

    # If team not found
        if home_team_id is None or away_team_id is None:
            #print(f"Team not found: {home_team_name} or {away_team_name}")
            continue

        home_team_id = home_team_id[0]
        away_team_id = away_team_id[0]
        # Insert into Player_Season
        cur.execute("""
            INSERT INTO Game (season_id, game_id, home_team_id, away_team_id, home_team_name, away_team_name, score_home_team, score_away_team)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (SEASON_ID, game_id, home_team_id, away_team_id, home_team_name, away_team_name, score_home_team, score_away_team))
        
        #print(f"Inserted {home_team_name}")
        game_id = game_id+1

        conn.commit()
        
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()


conn.commit()
cur.close()
conn.close()

In [ ]:
#########################
##ubacujem player_game##
#########################

In [ ]:
import time
import psycopg2
import pandas as pd

In [ ]:
DB_CONFIG = {
    "dbname" : "nba_sql4",
    "user" : "postgres",
    "password" : "1234",
    "host" : "localhost",
    "port" : "5432"
}

SEASON_ID = 1

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

In [ ]:
#df = pd.read_csv('PlayerStatistics.csv', delimiter=',', low_memory=False)  
#filtrirat samo utakmice u sezoni koju gledamo

#input_file = "PlayerStatistics.csv"   # Input CSV file
#output_file = "filtered_regular_season.csv"  # Output file
#start_date = pd.to_datetime("2022-10-18")
#end_date = pd.to_datetime("2023-04-10")

#df["gameDate"] = pd.to_datetime(df["gameDate"], errors="coerce")

#mask = (df["gameDate"] >= start_date) & (df["gameDate"] <= end_date)
#filtered_df = df.loc[mask]

#filtered_df.to_csv(output_file, index=False)

#print(f"Filtered data saved to {output_file}")
#print(f"Rows before: {len(df)}, Rows after filtering: {len(filtered_df)}")

In [ ]:
df = pd.read_csv('filtered_regular_season.csv', delimiter=',')

In [ ]:
for idx,row in df.iterrows():
    player_name = row['firstName'] + ' ' + row['lastName']
    
    cur.execute("SELECT player_id FROM PLAYER WHERE name = %s;", (player_name,))
    player_id = cur.fetchone()
    if player_id is None:
        continue

    game_id = row['gameId']

    pts = row['points']
    trb = row['reboundsTotal']
    ast = row['assists']
    fg_pct = row['fieldGoalsPercentage']
    fg3_pct = row['threePointersPercentage']
    ft_pct = row['freeThrowsPercentage']

    
    try:
        cur.execute("""
            INSERT INTO Player_Game (player_id, game_id, pts, trb, ast, fg_pct, fg3_pct, ft_pct)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        """, (player_id, game_id, pts, trb, ast, fg_pct, fg3_pct, ft_pct))

        #print(f"Inserted {player_id}")


    except Exception as e:
        #print(f"Error: {e}")
        conn.rollback()

conn.commit()
cur.close()
conn.close()

In [ ]:
#unosenje Team_Season podataka
import psycopg2
import pandas as pd

In [ ]:
DB_CONFIG = {
    "dbname" : "nba_sql4",
    "user" : "postgres",
    "password" : "1234",
    "host" : "localhost",
    "port" : "5432"
}

SEASON_ID = 1

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

df = pd.read_csv('team_season.csv', delimiter=',')

In [ ]:
for idx,row in df.iterrows():
    team_name = row['Team']

    cur.execute("SELECT team_id FROM TEAM WHERE name = %s;", (team_name,))
    team_id = cur.fetchone()

    wins, losses = row['Overall'].split('-')

    try:
        cur.execute("""
            INSERT INTO Team_Season (team_id, season_id, wins, losses)
            VALUES (%s,%s,%s,%s)
        """, (team_id, SEASON_ID, wins, losses))

        print(f"Inserted {team_name}")


    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()


# team_id , season_id , wins, losses

conn.commit()
cur.close()
conn.close()